> **Note:** This notebook reuses patterns inspired by the Azure AI Gateway Labs (https://github.com/Azure-Samples/AI-Gateway).

# 🤝 Citadel A2A Agent - Publish & Consume

## Run, publish, and call a Foundry Agent-to-Agent (A2A) agent end-to-end!

This notebook is fully reproducible from your current **azd environment** (`azd up` must have completed). It walks through the full lifecycle of running an agent on Microsoft Foundry, publishing it through the Citadel Governance Hub (APIM), and calling it as a governed A2A endpoint:

1. **Resolves** the agent-hosting Foundry account (the `-0` account) and its project from azd — no hardcoded names.
2. **Enables public network access** on that Foundry account so this notebook can reach its data plane.
3. **Creates** a simple HR **prompt agent** (mocked HR policy data) on `gpt-4.1`.
4. **Enables** the agent's incoming **A2A endpoint** and **verifies** its agent card.
5. **Re-disables public network access** to restore the secure, private-only posture.
6. **Publishes** the agent through **APIM** as a governed **access contract** (product + subscription).
7. **Calls** the agent through APIM with just a subscription key — proving APIM reaches it over private endpoints even with public access disabled.

Reference: [Enable an agent-to-agent endpoint](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint?tabs=rest-powershell%2Cverify-bash%2Cconnection-bash) and [Import an A2A agent API](https://learn.microsoft.com/en-us/azure/api-management/agent-to-agent-api).

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub and sample spoke. If you haven't done so, please follow the [Workshop Lab Guide](readme.md) (see [Lab 1 — Deploy Citadel to Your Azure Subscription](readme.md#lab-1)) before proceeding.

## Prerequisites

- Workshop Python environment set up per the [readme](readme.md#51-set-up-python-environment) (`uv sync` or
  `pip install -r requirements.txt`), which installs all dependencies this notebook uses.
- Azure CLI (`az`) logged in (`az login`) and `azd` initialized for the target environment.
- The Citadel hub deployed by `azd up` (APIM + two Foundry accounts).
- Permission to manage the Foundry account, create agents, and deploy to the hub resource group.


## 1. Configuration

This only sets the **agent identity** and **agent-card metadata**. All Azure resource names (Foundry account,
project, model, APIM) are resolved from your `azd` environment in later cells, so nothing here is environment-
specific. Override any default with the matching `A2A_*` / `AGENT_CARD_*` environment variable.

| Setting | Purpose |
| --- | --- |
| `AGENT_NAME` | Name of the prompt agent this notebook creates and publishes. |
| `AGENT_*` / `SKILL_*` | Agent card metadata published to A2A callers. |
| `VERIFY_AGENT_CARD` | Whether to GET and validate the v1.0 card after enabling. |


In [ ]:
import os
from urllib.parse import quote

# --- Agent identity ------------------------------------------------------
# Name of the prompt agent this notebook creates and then publishes via A2A.
AGENT_NAME = os.environ.get("A2A_AGENT_NAME", "hr-assistant")

# --- Agent card metadata (published to A2A callers) ----------------------
AGENT_DESCRIPTION = os.environ.get(
    "AGENT_CARD_DESCRIPTION",
    "Contoso HR assistant answering HR policy questions from mocked data",
)
AGENT_VERSION = os.environ.get("AGENT_CARD_VERSION", "1.0")
SKILL_ID = os.environ.get("AGENT_CARD_SKILL_ID", "hr-policy-qa")
SKILL_NAME = os.environ.get("AGENT_CARD_SKILL_NAME", "HR Policy Q&A")
SKILL_DESCRIPTION = os.environ.get(
    "AGENT_CARD_SKILL_DESCRIPTION",
    "Answers HR questions about leave, benefits, and payroll using mocked policy data",
)

# --- Behavior flags ------------------------------------------------------
VERIFY_AGENT_CARD = os.environ.get("VERIFY_AGENT_CARD", "true").lower() == "true"
SHOW_AGENT_CARD = os.environ.get("SHOW_AGENT_CARD", "true").lower() == "true"

# --- API version ---------------------------------------------------------
AGENT_API_VERSION = os.environ.get("AGENT_API_VERSION", "v1")

ESCAPED_AGENT_NAME = quote(AGENT_NAME, safe="")


def _print_config():
    rows = [
        ("Agent name", AGENT_NAME),
        ("Description", AGENT_DESCRIPTION),
        ("Card version", AGENT_VERSION),
        ("Skill", f"{SKILL_NAME} ({SKILL_ID})"),
        ("Skill details", SKILL_DESCRIPTION),
        ("Agent API version", AGENT_API_VERSION),
        ("Verify agent card", "yes" if VERIFY_AGENT_CARD else "no"),
        ("Show agent card", "yes" if SHOW_AGENT_CARD else "no"),
    ]
    label_w = max(len(label) for label, _ in rows)

    print("=" * 70)
    print("🤖  A2A AGENT CONFIGURATION")
    print("=" * 70)
    for label, value in rows:
        print(f"  {label.rjust(label_w)} : {value}")
    print("=" * 70)
    print("ℹ️  All Azure resource names (Foundry account, project, model, APIM) are")
    print("   resolved from your azd environment in the next cells — nothing here")
    print("   is environment-specific.")


_print_config()


## 2. Helpers

Small utilities:

- `get_access_token(...)` wraps `az account get-access-token` (data-plane resource **or** management scope).
- `get_azd_value(...)` reads a value from the `azd` environment for auto-resolution.
- `join_uri_path(...)` joins URL segments safely.


In [ ]:
import shutil
import subprocess
import requests


def _run(cmd):
    """Run a CLI command and return trimmed stdout, or None on failure."""
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
    except FileNotFoundError:
        return None
    if result.returncode != 0:
        return None
    out = (result.stdout or "").strip()
    return out or None


def get_access_token(resource=None, scope=None):
    """Acquire an Azure AD access token via the Azure CLI."""
    if resource:
        cmd = ["az", "account", "get-access-token", "--resource", resource, "--query", "accessToken", "-o", "tsv"]
    else:
        cmd = ["az", "account", "get-access-token", "--scope", scope, "--query", "accessToken", "-o", "tsv"]
    token = _run(cmd)
    if not token:
        raise RuntimeError("Failed to get an access token. Is the Azure CLI logged in (az login)?")
    return token


def get_azd_value(name):
    """Read a value from the azd environment, or None if unavailable."""
    if not shutil.which("azd"):
        return None
    return _run(["azd", "env", "get-value", name])


def join_uri_path(root, path):
    return f"{root.rstrip('/')}/{path.lstrip('/')}"


# Confirm the Azure CLI is logged in (equivalent to `az account show`).
if _run(["az", "account", "show", "--query", "id", "-o", "tsv"]) is None:
    raise RuntimeError("Azure CLI is not logged in. Run `az login`, then re-run this cell.")
print("Azure CLI session OK.")


## 3. Resolve the Foundry environment from azd

Everything below is derived from your current `azd` environment — no hardcoded resource names. We resolve
the **agent-hosting Foundry account** (the `-0` account from `LLM_BACKEND_CONFIG`), its **project**, the
**model deployment** (`gpt-4.1`), and build the project **endpoint** (`BASE_URL`). Override any value with the
matching `A2A_*` environment variable if needed.


In [ ]:
import json

RESOURCE_GROUP = os.environ.get("AZURE_RESOURCE_GROUP") or get_azd_value("AZURE_RESOURCE_GROUP")
SUBSCRIPTION_ID = os.environ.get("AZURE_SUBSCRIPTION_ID") or get_azd_value("AZURE_SUBSCRIPTION_ID")
if not RESOURCE_GROUP or not SUBSCRIPTION_ID:
    raise ValueError("Could not resolve AZURE_RESOURCE_GROUP / AZURE_SUBSCRIPTION_ID from azd. Run `azd env refresh`.")

# Agents are hosted on the "-0" Foundry account. Resolve it from LLM_BACKEND_CONFIG (or override).
FOUNDRY_ACCOUNT_NAME = os.environ.get("A2A_FOUNDRY_ACCOUNT_NAME")
if not FOUNDRY_ACCOUNT_NAME:
    backends = json.loads(get_azd_value("LLM_BACKEND_CONFIG") or "[]")
    zero = [b for b in backends if str(b.get("backendId", "")).endswith("-0")]
    if not zero:
        raise ValueError("No '-0' Foundry backend found in LLM_BACKEND_CONFIG; set A2A_FOUNDRY_ACCOUNT_NAME.")
    FOUNDRY_ACCOUNT_NAME = zero[0]["backendId"]

mgmt_token = get_access_token(scope="https://management.azure.com/.default")
arm_account = (
    f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
    f"/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_ACCOUNT_NAME}"
)

# Discover the project on that account (first project unless overridden).
FOUNDRY_PROJECT_NAME = os.environ.get("A2A_FOUNDRY_PROJECT_NAME")
if not FOUNDRY_PROJECT_NAME:
    projects = requests.get(
        f"{arm_account}/projects?api-version=2025-04-01-preview",
        headers={"Authorization": f"Bearer {mgmt_token}"},
    ).json().get("value", [])
    if not projects:
        raise ValueError(f"No projects found on {FOUNDRY_ACCOUNT_NAME}; set A2A_FOUNDRY_PROJECT_NAME.")
    FOUNDRY_PROJECT_NAME = projects[0]["name"].split("/")[-1]

MODEL_DEPLOYMENT = os.environ.get("A2A_MODEL_DEPLOYMENT", "gpt-4.1")
BASE_URL = f"https://{FOUNDRY_ACCOUNT_NAME}.services.ai.azure.com/api/projects/{FOUNDRY_PROJECT_NAME}"

print(f"Resource group  : {RESOURCE_GROUP}")
print(f"Foundry account : {FOUNDRY_ACCOUNT_NAME}")
print(f"Foundry project : {FOUNDRY_PROJECT_NAME}")
print(f"Model deployment: {MODEL_DEPLOYMENT}")
print(f"Project endpoint: {BASE_URL}")


## 4. Enable public network access on the agent's Foundry account

After `azd up`, the `-0` Foundry account has **public network access disabled**, so this notebook can't reach
its data plane to create an agent or enable A2A. We enable it temporarily here and **re-disable it in step 9**
once the agent and its A2A endpoint are ready.


In [ ]:
import time


def set_foundry_public_network_access(enabled: bool):
    """Toggle public network access on the agent's Foundry account via ARM."""
    token = get_access_token(scope="https://management.azure.com/.default")
    body = {
        "properties": {
            "publicNetworkAccess": "Enabled" if enabled else "Disabled",
            "networkAcls": {"defaultAction": "Allow" if enabled else "Deny"},
        }
    }
    r = requests.patch(
        f"{arm_account}?api-version=2024-10-01",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=body,
    )
    r.raise_for_status()
    props = r.json().get("properties", {})
    return props.get("publicNetworkAccess"), (props.get("networkAcls") or {}).get("defaultAction")


pna, acl = set_foundry_public_network_access(True)
print(f"Public network access: {pna} (default action: {acl})")
print("Waiting ~30s for the network change to propagate...")
time.sleep(30)


## 5. Create the HR prompt agent

Create a simple **prompt agent** on `gpt-4.1` that answers HR questions using **mocked policy data** baked
into its instructions. This uses the Foundry Agents SDK (`PromptAgentDefinition`), the same pattern as
[4. citadel-agent-frameworks-tests.ipynb](4.%20citadel-agent-frameworks-tests.ipynb). A quick smoke test
confirms the agent responds before we expose it over A2A.


In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

HR_INSTRUCTIONS = """You are Contoso's HR assistant. Answer employee HR questions using ONLY the mocked
company policy data below. Be concise and friendly. If a question is outside this data, say you don't have
that information and suggest contacting hr@contoso.example.

MOCKED CONTOSO HR POLICY DATA
- Annual leave: 25 paid vacation days per year, accrued monthly, up to 5 days carry-over.
- Sick leave: 10 paid sick days per year; a doctor's note is required after 3 consecutive days.
- Parental leave: 16 weeks paid for the primary caregiver, 6 weeks for the secondary caregiver.
- Health benefits: Contoso covers 90% of the medical premium and 100% of basic dental/vision.
- Retirement: 401(k) with a 6% company match, vesting over 3 years.
- Payroll: paid on the 15th and the last business day of each month.
- Open enrollment: runs the first two weeks of November each year.
- Remote work: up to 3 days per week remote with manager approval.
- Expense reimbursement: submit within 30 days via the MyExpense portal; meals capped at $75/day.
"""

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=BASE_URL, credential=credential)

# Create the agent. The Foundry network change may still be propagating, so retry briefly.
agent = None
last_err = None
for attempt in range(6):
    try:
        agent = project_client.agents.create_version(
            agent_name=AGENT_NAME,
            definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=HR_INSTRUCTIONS),
        )
        break
    except Exception as e:
        last_err = e
        print(f"  attempt {attempt + 1}/6 failed: {str(e)[:140]} — retrying in 15s")
        time.sleep(15)

if agent is None:
    raise RuntimeError(f"Failed to create agent after retries. Last error: {last_err}")

print(f"Agent created: name={agent.name} version={agent.version} id={agent.id}")

# Smoke test: ask one HR question (public network access is currently enabled).
with project_client.get_openai_client() as oai:
    conv = oai.conversations.create(
        items=[{"type": "message", "role": "user",
                "content": "How many vacation days do I get and can I carry any over?"}],
    )
    resp = oai.responses.create(
        conversation=conv.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
    )
    print("\nSmoke test reply:")
    print(resp.output_text if hasattr(resp, "output_text") else str(resp))


## 6. Enable the incoming A2A endpoint

PATCH the agent with an **agent card** (description, version, skills) and an **agent endpoint** that advertises
the `responses` and `a2a` protocols. This is a data-plane call against the Foundry project endpoint (requires
public network access, enabled in step 4) and uses an `https://ai.azure.com` token.


In [ ]:
agent_token = get_access_token(resource="https://ai.azure.com")

patch_uri = f"{BASE_URL}/agents/{ESCAPED_AGENT_NAME}?api-version={AGENT_API_VERSION}"

body = {
    "agent_card": {
        "description": AGENT_DESCRIPTION,
        "version": AGENT_VERSION,
        "skills": [
            {
                "id": SKILL_ID,
                "name": SKILL_NAME,
                "description": SKILL_DESCRIPTION,
            }
        ],
    },
    "agent_endpoint": {
        "protocols": ["responses", "a2a"],
    },
}

print(f"PATCH {patch_uri}")
response = requests.patch(
    patch_uri,
    headers={"Authorization": f"Bearer {agent_token}", "Content-Type": "application/json"},
    json=body,
)
response.raise_for_status()
print(f"Success ({response.status_code}). A2A endpoint enabled for agent '{AGENT_NAME}'.")

## 7. Resolve the agent card URLs

Once the endpoint is enabled, the agent publishes its card at well-known paths. The `v1.0` card is the current
spec; `v0.3` is also exposed for older A2A clients.


In [ ]:
a2a_base_path = join_uri_path(BASE_URL, f"agents/{ESCAPED_AGENT_NAME}/endpoint/protocols/a2a")
agent_card_v10_url = join_uri_path(a2a_base_path, "agentCard/v1.0")
agent_card_v03_url = join_uri_path(a2a_base_path, "agentCard/v0.3")

print(f"A2A base path  : {a2a_base_path}")
print(f"Agent card v1.0: {agent_card_v10_url}")
print(f"Agent card v0.3: {agent_card_v03_url}")

## 8. Verify the v1.0 agent card

GET the `v1.0` card with an authenticated request and confirm it reports a protocol version. Set
`SHOW_AGENT_CARD=true` (default) to print the full card. This is the last step that talks to Foundry directly
from the notebook — after this we re-disable public access.


In [ ]:
import json

if VERIFY_AGENT_CARD:
    card_response = requests.get(
        agent_card_v10_url,
        headers={"Authorization": f"Bearer {agent_token}"},
    )
    card_response.raise_for_status()
    agent_card = card_response.json()

    # The protocol version may be top-level or nested under supportedInterfaces.
    protocol_version = agent_card.get("protocolVersion")
    if not protocol_version:
        interfaces = agent_card.get("supportedInterfaces") or []
        versions = sorted({i.get("protocolVersion") for i in interfaces if i.get("protocolVersion")})
        protocol_version = ", ".join(versions) if versions else None

    if protocol_version:
        print(f"Verified agent '{agent_card.get('name')}' — A2A protocol version(s): {protocol_version}")
    else:
        print("WARNING: the agent card response did not include a protocol version.")

    if SHOW_AGENT_CARD:
        print(json.dumps(agent_card, indent=2))
else:
    print("Verification skipped (VERIFY_AGENT_CARD is false).")


### Agent is A2A-ready

The agent's incoming A2A endpoint is enabled and its v1.0 card is verified. Next we restore the secure posture
(step 9), then publish and call it through APIM (steps 10–11).


In [ ]:
print("A2A endpoint setup completed.")

## 9. Re-disable public network access (restore secure posture)

The agent now has its A2A endpoint enabled and its card verified. Restore the original **private-only** posture
by disabling public network access on the Foundry account. The notebook can no longer reach the Foundry data
plane directly after this — but **APIM can**, over its private endpoint, which is exactly what the next sections
use.


In [ ]:
pna, acl = set_foundry_public_network_access(False)
print(f"Public network access: {pna} (default action: {acl})")
print("Foundry is now private-only. APIM will reach it via its private endpoint.")


---
## 10. Expose the agent as an A2A agent through APIM (access-contract pattern)

The cells above created the agent and enabled its **incoming** A2A endpoint on Foundry. Now we publish that
endpoint through the **Citadel Governance Hub (APIM)** and govern it with an **access contract** — the same
product + subscription pattern from [3. citadel-access-contracts-tests.ipynb](3.%20citadel-access-contracts-tests.ipynb).

What these cells do:

1. **Create an APIM API** that fronts the agent's A2A JSON-RPC runtime URL, with operations for the JSON-RPC
   endpoint and the agent card.
2. **Attach an API policy** so APIM authenticates to the Foundry backend using its **managed identity**
   (audience `https://ai.azure.com`).
3. **Create an access contract** (APIM **product** + **subscription**) bound to that API, reusing the repo's
   `apimOnboardService.bicep` module — so callers need a subscription key, and you get governance
   (rate-limiting, etc.) at the product scope.
4. **Call the agent through APIM** using the subscription key (agent card `GET` + JSON-RPC `message/send`).

> **About the native "A2A Agent" API type:** the Azure portal offers a first-class *A2A Agent* import that
> rewrites the agent-card host and forces JSON-RPC transport
> ([docs](https://learn.microsoft.com/en-us/azure/api-management/agent-to-agent-api)). That import path is
> portal-only today. To keep this notebook fully reproducible over REST, we front the same backend as a
> **governed pass-through** and wrap it in an access contract — the calling experience (subscription key +
> JSON-RPC) is the same.

> **Why this works with public access disabled:** APIM is VNet-integrated and reaches the Foundry account
> over a **private endpoint**, so steps 10–11 succeed even though we turned public access off in step 9. The
> APIM managed identity needs an Azure AI role (e.g. *Cognitive Services User*) on the agent's Foundry account
> — `azd up` already grants this for the hub APIM.


### 10.1 Resolve APIM coordinates

Initialize the shared `APIMClientTool` against the hub resource group (the same `AZURE_RESOURCE_GROUP` that
holds the agent's Foundry), and compute the agent's JSON-RPC runtime URL — the backend APIM will front.


In [ ]:
import sys

sys.path.insert(1, "../shared")  # make the shared helpers importable
from apimtools import APIMClientTool  # noqa: E402

# Publish through the APIM in the hub resource group (same azd env as the agent's Foundry).
# Override with A2A_HUB_RESOURCE_GROUP only if your APIM lives in a different RG.
GOV_HUB_RG = os.environ.get("A2A_HUB_RESOURCE_GROUP") or RESOURCE_GROUP

# Discover APIM in that resource group (name, gateway URL, subscription id).
apim_tool = APIMClientTool(GOV_HUB_RG)
apim_tool.initialize()

APIM_NAME = str(apim_tool.apim_resource_name)
APIM_GATEWAY_URL = str(apim_tool.apim_resource_gateway_url).rstrip("/")
HUB_SUBSCRIPTION_ID = str(apim_tool.subscription_id)

# The agent's JSON-RPC A2A runtime URL = the backend APIM will front.
AGENT_A2A_RUNTIME_URL = join_uri_path(BASE_URL, f"agents/{ESCAPED_AGENT_NAME}/endpoint/protocols/a2a")

# APIM API + access-contract identifiers (override via env if you like).
A2A_API_NAME = os.environ.get("A2A_APIM_API_NAME", f"{AGENT_NAME}-a2a")
A2A_API_PATH = os.environ.get("A2A_APIM_API_PATH", f"{AGENT_NAME}-a2a")
A2A_API_DISPLAY = os.environ.get("A2A_APIM_API_DISPLAY", f"{AGENT_NAME} A2A Agent")
A2A_PRODUCT_ID = os.environ.get("A2A_PRODUCT_ID", f"A2A-{AGENT_NAME}-DEV")

print(f"APIM resource group : {GOV_HUB_RG}")
print(f"APIM name           : {APIM_NAME}")
print(f"APIM gateway URL    : {APIM_GATEWAY_URL}")
print(f"Agent backend (A2A) : {AGENT_A2A_RUNTIME_URL}")
print(f"APIM A2A API        : {A2A_API_NAME}  (path: /{A2A_API_PATH})")
print(f"Access contract     : {A2A_PRODUCT_ID}")


### 10.2 Create the APIM API fronting the agent's A2A endpoint

PUT an APIM API whose backend (`serviceUrl`) is the agent's A2A runtime URL, and add operations for the
JSON-RPC root (`POST /`) and the agent card (`GET /agentCard/v1.0`, `GET /agentCard/v0.3`). The API requires a
subscription key, so it can only be reached through an access-contract subscription.


In [ ]:
APIM_MGMT_API_VERSION = os.environ.get("APIM_MGMT_API_VERSION", "2024-10-01-preview")

mgmt_token = get_access_token(scope="https://management.azure.com/.default")
mgmt_headers = {"Authorization": f"Bearer {mgmt_token}", "Content-Type": "application/json"}

apim_base = (
    f"https://management.azure.com/subscriptions/{HUB_SUBSCRIPTION_ID}"
    f"/resourceGroups/{GOV_HUB_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}"
)

# 1) Upsert the API that fronts the agent's A2A JSON-RPC backend.
api_uri = f"{apim_base}/apis/{A2A_API_NAME}?api-version={APIM_MGMT_API_VERSION}"
api_body = {
    "properties": {
        "displayName": A2A_API_DISPLAY,
        "path": A2A_API_PATH,
        "protocols": ["https"],
        "serviceUrl": AGENT_A2A_RUNTIME_URL,
        "subscriptionRequired": True,
        "subscriptionKeyParameterNames": {
            "header": "Ocp-Apim-Subscription-Key",
            "query": "subscription-key",
        },
    }
}
r = requests.put(api_uri, headers=mgmt_headers, json=api_body)
r.raise_for_status()
print(f"API upserted: {A2A_API_NAME} ({r.status_code})")

# 2) Operations: JSON-RPC POST at the root + agent-card GETs.
operations = {
    "a2a-jsonrpc": {"displayName": "A2A JSON-RPC", "method": "POST", "urlTemplate": "/"},
    "get-agent-card-v1": {"displayName": "Agent card v1.0", "method": "GET", "urlTemplate": "/agentCard/v1.0"},
    "get-agent-card-v03": {"displayName": "Agent card v0.3", "method": "GET", "urlTemplate": "/agentCard/v0.3"},
}
for op_id, op in operations.items():
    op_uri = f"{apim_base}/apis/{A2A_API_NAME}/operations/{op_id}?api-version={APIM_MGMT_API_VERSION}"
    op_body = {"properties": {**op, "templateParameters": [], "responses": []}}
    ro = requests.put(op_uri, headers=mgmt_headers, json=op_body)
    ro.raise_for_status()
    print(f"  operation: {op_id} -> {op['method']} {op['urlTemplate']} ({ro.status_code})")

print("\nAPIM API and operations are ready.")


### 10.3 Attach the backend-auth API policy

The agent's A2A endpoint requires a bearer token (audience `https://ai.azure.com`). This policy makes APIM mint
that token from its **managed identity** and forward it to the backend, so callers only need a subscription
key — never a Foundry token.

> The APIM managed identity must hold an Azure AI role (e.g. *Cognitive Services User*) on the agent's Foundry
> account. `azd up` already grants this for the hub APIM. This notebook auto-detects whether the identity is
> system- or user-assigned and pins the `client-id` when needed.


In [ ]:
# Detect APIM's managed identity so the backend-auth policy targets the right one.
# - System-assigned: <authentication-managed-identity resource="..." />
# - User-assigned  : same, plus client-id="<clientId>"
identity = requests.get(
    f"{apim_base}?api-version={APIM_MGMT_API_VERSION}",
    headers=mgmt_headers,
).json().get("identity", {})

identity_type = (identity.get("type") or "").lower()
client_id_attr = ""
if "userassigned" in identity_type and not identity.get("principalId"):
    # No system-assigned identity available; pin to the first user-assigned identity.
    uami = identity.get("userAssignedIdentities") or {}
    first = next(iter(uami.values()), {})
    client_id = first.get("clientId")
    if not client_id:
        raise RuntimeError("APIM has a user-assigned identity but no clientId was returned.")
    client_id_attr = f' client-id="{client_id}"'
    print(f"Using user-assigned identity (client-id={client_id}).")
else:
    print("Using system-assigned identity.")

api_policy_xml = f"""<policies>
  <inbound>
    <base />
    <!-- APIM calls joker's A2A backend with a token from its managed identity. -->
    <authentication-managed-identity resource="https://ai.azure.com"{client_id_attr} />
  </inbound>
  <backend>
    <base />
  </backend>
  <outbound>
    <base />
  </outbound>
  <on-error>
    <base />
  </on-error>
</policies>"""

policy_uri = f"{apim_base}/apis/{A2A_API_NAME}/policies/policy?api-version={APIM_MGMT_API_VERSION}"
rp = requests.put(
    policy_uri,
    headers=mgmt_headers,
    json={"properties": {"format": "rawxml", "value": api_policy_xml}},
)
rp.raise_for_status()
print(f"API policy applied ({rp.status_code}).")
print("Backend auth: APIM managed identity -> https://ai.azure.com")


### 10.4 Create the access contract (product + subscription)

Now wrap the API in an **access contract** using the repo's `apimOnboardService.bicep` module — the same
building block the Citadel access-contracts notebook deploys. It creates a published **product** bound to the
A2A API, applies a product-scope governance policy (rate-limiting), and issues a **subscription** whose key
callers use. We generate a `.bicepparam` and deploy it at resource-group scope.


In [ ]:
import re

# Reuse the repo's access-contract building block (product + subscription bound to the API).
_slug = re.sub(r"[^a-z0-9]+", "-", AGENT_NAME.lower()).strip("-")
contract_dir = f"../bicep/infra/citadel-access-contracts/contracts/a2a-{_slug}/dev"
os.makedirs(contract_dir, exist_ok=True)

# Product-scope governance policy (kept simple: subscription key is enforced by the product).
product_policy_xml = """<policies>
  <inbound>
    <base />
    <rate-limit calls="3" renewal-period="60" />
  </inbound>
  <backend>
    <base />
  </backend>
  <outbound>
    <base />
  </outbound>
  <on-error>
    <base />
  </on-error>
</policies>"""
with open(os.path.join(contract_dir, "ai-a2a-product-policy.xml"), "w") as f:
    f.write(product_policy_xml)

subscription_name = f"{A2A_PRODUCT_ID}-SUB-01"
bicepparam = f"""using '../../../modules/apimOnboardService.bicep'

param apimName = '{APIM_NAME}'
param productId = '{A2A_PRODUCT_ID}'
param productDisplayName = '{A2A_API_DISPLAY} Access Contract'
param productDescription = 'A2A access contract for agent {AGENT_NAME} (created from notebook)'
param productTerms = 'A2A access contract created from the enable-a2a-endpoint notebook.'
param apiNames = ['{A2A_API_NAME}']
param productPolicyXml = loadTextContent('ai-a2a-product-policy.xml')
param subscriptionName = '{subscription_name}'
param subscriptionDisplayName = '{subscription_name}'
"""
param_file = os.path.join(contract_dir, "main.bicepparam")
with open(param_file, "w") as f:
    f.write(bicepparam)

deployment_name = f"a2a-access-contract-{time.strftime('%Y%m%d%H%M%S')}"
cmd = [
    "az", "deployment", "group", "create",
    "--name", deployment_name,
    "--resource-group", GOV_HUB_RG,
    "--parameters", param_file,
    "-o", "json",
]
print(f"Deploying access contract '{A2A_PRODUCT_ID}' (product + subscription)...")
proc = subprocess.run(cmd, capture_output=True, text=True)
if proc.returncode != 0:
    raise RuntimeError(f"Deployment failed:\n{proc.stderr}")

# Fetch the subscription's primary key directly from APIM (reliable for secrets).
secrets_uri = (
    f"{apim_base}/subscriptions/{subscription_name}/listSecrets"
    f"?api-version={APIM_MGMT_API_VERSION}"
)
secrets_resp = requests.post(secrets_uri, headers=mgmt_headers)
secrets_resp.raise_for_status()
A2A_SUBSCRIPTION_KEY = secrets_resp.json()["primaryKey"]

# The API path is what we set when creating the API in section 10.2.
APIM_A2A_RUNTIME_BASE = join_uri_path(APIM_GATEWAY_URL, A2A_API_PATH)

print(f"Access contract ready : {A2A_PRODUCT_ID}")
print(f"Subscription          : {subscription_name}")
print(f"APIM runtime base URL : {APIM_A2A_RUNTIME_BASE}")
print(f"Subscription key      : {A2A_SUBSCRIPTION_KEY[:6]}... (stored in A2A_SUBSCRIPTION_KEY)")


---
## 11. Call the agent through the APIM A2A endpoint

With the access contract in place, callers reach the agent via APIM using only the **subscription key** — and
this works even though public network access on the Foundry account is **disabled** (step 9), because APIM
reaches it over a private endpoint. We do two calls:

1. **Discover** — `GET` the agent card through APIM.
2. **Invoke** — `POST` an A2A JSON-RPC `message/send` request and read the agent's reply.


In [ ]:
# 1) Discover: fetch the agent card through APIM using only the subscription key.
apim_card_url = join_uri_path(APIM_A2A_RUNTIME_BASE, "agentCard/v1.0")
print(f"GET {apim_card_url}")

card_resp = requests.get(
    apim_card_url,
    headers={"Ocp-Apim-Subscription-Key": A2A_SUBSCRIPTION_KEY},
)
print(f"Status: {card_resp.status_code}")
if card_resp.ok:
    card_json = card_resp.json()
    print(f"Agent name      : {card_json.get('name')}")
    print(f"protocolVersion : {card_json.get('protocolVersion')}")
    print(json.dumps(card_json, indent=2)[:1500])
else:
    print(card_resp.text[:1000])


In [ ]:
import uuid

apim_headers = {
    "Ocp-Apim-Subscription-Key": A2A_SUBSCRIPTION_KEY,
    "Content-Type": "application/json",
}


def extract_text(result):
    """Pull any text parts out of an A2A task/message result."""
    texts = []
    for part in (result.get("parts") or []):
        if part.get("kind") == "text" and part.get("text"):
            texts.append(part["text"])
    for artifact in (result.get("artifacts") or []):
        for part in (artifact.get("parts") or []):
            if part.get("kind") == "text" and part.get("text"):
                texts.append(part["text"])
    for msg in (result.get("history") or []):
        for part in (msg.get("parts") or []):
            if part.get("kind") == "text" and part.get("text"):
                texts.append(part["text"])
    status_msg = (result.get("status") or {}).get("message") or {}
    for part in (status_msg.get("parts") or []):
        if part.get("kind") == "text" and part.get("text"):
            texts.append(part["text"])
    return "\n".join(texts)


# Invoke: send an A2A JSON-RPC message/send request through APIM.
a2a_request = {
    "jsonrpc": "2.0",
    "id": str(uuid.uuid4()),
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "How much parental leave does the primary caregiver get, and when is open enrollment?"}],
            "messageId": str(uuid.uuid4()),
            "kind": "message",
        }
    },
}

print(f"POST {APIM_A2A_RUNTIME_BASE}  (message/send)")
invoke_resp = requests.post(APIM_A2A_RUNTIME_BASE, headers=apim_headers, json=a2a_request)
print(f"Status: {invoke_resp.status_code}")
invoke_resp.raise_for_status()
result = invoke_resp.json().get("result", {})

# message/send may return a final message or an async task. Poll the task if needed.
task_id = result.get("id") if result.get("kind") == "task" else None
state = (result.get("status") or {}).get("state")

for _ in range(20):
    if result.get("kind") == "message" or state in ("completed", "failed", "canceled", "rejected"):
        break
    time.sleep(2)
    poll = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "tasks/get",
        "params": {"id": task_id},
    }
    poll_resp = requests.post(APIM_A2A_RUNTIME_BASE, headers=apim_headers, json=poll)
    poll_resp.raise_for_status()
    result = poll_resp.json().get("result", {})
    state = (result.get("status") or {}).get("state")
    print(f"  task state: {state}")

reply = extract_text(result)
print("\n--- agent says (via APIM) ---")
print(reply if reply else json.dumps(result, indent=2)[:1500])


### Notes & troubleshooting

- **`500` with backend "name is valid, but no data of the requested type was found"**: a DNS/network error —
  APIM can't reach the agent's Foundry host. Ensure you publish through the APIM in the **same** azd
  environment as the agent (its VNet has a private endpoint to that Foundry). `GOV_HUB_RG` defaults to
  `AZURE_RESOURCE_GROUP`; override with `A2A_HUB_RESOURCE_GROUP` only if your APIM is elsewhere.
- **`500` with a managed-identity error in the trace**: APIM had no usable identity. This notebook auto-detects
  system- vs user-assigned identity (section 10.3); if user-assigned, it pins the `client-id`.
- **`401/403` from the backend** (after a token is obtained): the APIM managed identity lacks an Azure AI
  role on the agent's Foundry account. Grant it **Cognitive Services User** (or your tenant's equivalent
  Azure AI role) on that account and retry. Role propagation can take a few minutes.
- **`401` at APIM** (before the backend): missing/invalid `Ocp-Apim-Subscription-Key`, or the subscription
  isn't active. Re-run section 10.4.
- **`403 PublicNetworkAccess` when creating the agent (step 5)**: the network change from step 4 hasn't
  propagated yet. The create cell retries; if it still fails, wait a minute and re-run step 5.
- **`404` on the agent card**: confirm the agent's A2A endpoint is enabled (steps 6–8) and that `BASE_URL`
  matches your project/agent.
- **Reuse for other agents**: change `A2A_AGENT_NAME` (and optionally `A2A_PRODUCT_ID`) and re-run from
  step 5 to create and publish another agent under its own access contract.


<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the A2A access contract (APIM product + subscription) and the A2A API created during this notebook
session.

> **Note:** This is **disabled by default**. Set `cleanup_enabled = True` in the next cell to remove the
> resources. This does not delete the Foundry agent itself (uncomment the optional block to do that, which
> requires re-enabling public network access).


In [ ]:
# Set to True to delete the A2A access contract and API created in this session
cleanup_enabled = False

if cleanup_enabled:
    # Remove the access-contract product (and its subscriptions)
    utils_cmd = [
        "az", "apim", "product", "delete",
        "-g", GOV_HUB_RG, "--service-name", APIM_NAME,
        "--product-id", A2A_PRODUCT_ID, "--delete-subscriptions", "true", "--yes",
    ]
    print(f"Deleting product '{A2A_PRODUCT_ID}' and its subscriptions...")
    subprocess.run(utils_cmd, check=False)

    # Remove the A2A API
    print(f"Deleting API '{A2A_API_NAME}'...")
    subprocess.run(
        ["az", "apim", "api", "delete", "-g", GOV_HUB_RG, "--service-name", APIM_NAME,
         "--api-id", A2A_API_NAME, "--yes"],
        check=False,
    )

    # Optionally delete the agent version from Foundry (requires public access re-enabled).
    # set_foundry_public_network_access(True)
    # project_client.agents.delete_version(agent_name=AGENT_NAME, agent_version=agent.version)

    print("Cleanup completed!")
else:
    print("Cleanup is disabled. Set cleanup_enabled = True to remove the A2A access contract and API.")


Deleting product 'A2A-hr-assistant-DEV' and its subscriptions...
Deleting API 'hr-assistant-a2a'...
Cleanup completed!
